## Prepare Data

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
import torch
print(torch.cuda.device_count())

In [ ]:
from datasets import load_dataset, DownloadConfig
import os

download_config = DownloadConfig(
    local_files_only=True,
    cache_dir=".cache",   # optional
)

In [ ]:
dataset = load_dataset("fleurs", 'af_za', streaming=True, download_config=download_config, trust_remote_code=True)

In [ ]:
def prepare_batch(batch):
    audios = [x["array"][:360000] for x in batch["audio"]]
    sampling_rate = batch["audio"][0]["sampling_rate"]

    inputs = processor(
        audios,
        sampling_rate=sampling_rate,
        padding=False,   # IMPORTANT: no padding here
    )

    labels = processor.tokenizer(batch["ipa"]).input_ids

    return {
        "input_values": inputs["input_values"],
        "attention_mask": inputs["attention_mask"],
        "labels": labels,
    }

In [ ]:
dataset = dataset.map(prepare_batch, batched=True, batch_size=16, remove_columns=dataset['train'].column_names)

## Configure Model

In [ ]:
from transformers import Wav2Vec2CTCTokenizer

In [ ]:
tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file="fleurs_ipa_asr/vocab.json",
    word_delimiter_token='|',
    unk_token="<unk>",
    pad_token="<pad>",
    bos_token="<s>",
    eos_token="</s>",
    model_max_length=1024,
)

In [ ]:
from transformers import Wav2Vec2Processor, Wav2Vec2FeatureExtractor

MODEL_STR =  "facebook/wav2vec2-lv-60-espeak-cv-ft"

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_STR)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)


In [ ]:
from transformers import Wav2Vec2ForCTC, Wav2Vec2Config

config = Wav2Vec2Config.from_pretrained(MODEL_STR)
config.pad_token_id = tokenizer.pad_token_id
config.bos_token_id = tokenizer.bos_token_id
config.eos_token_id = tokenizer.eos_token_id
config.vocab_size = tokenizer.vocab_size

model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_STR,
    config=config,
    ignore_mismatched_sizes=True,
)

In [ ]:
model.config.ctc_zero_infinity = True
model.config.ctc_loss_reduction = "mean"

In [ ]:
model.save_pretrained("models/w2v2-lv-60-espeak-ipa")
processor.save_pretrained("models/w2v2-lv-60-espeak-ipa")

## Training

In [4]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC, Wav2Vec2Config

MODEL_DIR = "models/w2v2-lv-60-espeak-ipa/"
processor = Wav2Vec2Processor.from_pretrained(MODEL_DIR)
config = Wav2Vec2Config.from_pretrained(MODEL_DIR)
config.adapter_attn_dim = 16
model = Wav2Vec2ForCTC.from_pretrained(MODEL_DIR, target_lang='af_za')
model.freeze_feature_encoder()

/localscratch/nnsfn01/anaconda3/envs/north_caucasus/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ValueError: Cannot pass `target_lang`: af_za if `config.adapter_attn_dim` is not defined.

In [ ]:
for name, param in model.named_parameters():
    param.requires_grad = ("adapter" in name) or ("lm_head" in name)
    
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(trainable / total)

In [ ]:
from dataclasses import dataclass
from typing import List, Dict
import torch

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor

    def __call__(self, features: List[Dict]):
        # separate inputs and labels
        input_features = [
            {
                "input_values": f["input_values"],
                "attention_mask": f["attention_mask"],
            }
            for f in features
        ]

        label_features = [{"input_ids": f["labels"]} for f in features]

        # pad audio
        batch = self.processor.pad(
            input_features,
            padding=True,
            return_tensors="pt",
        )

        # pad labels
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=True,
            return_tensors="pt",
        )

        # mask padding
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1), -100
        )

        batch["labels"] = labels
        return batch

In [ ]:
data_collator = DataCollatorCTCWithPadding(processor=processor)

In [ ]:
import numpy as np
import evaluate

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    # decode predictions
    pred_str = processor.batch_decode(pred_ids)

    # replace -100 in labels
    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # decode labels
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    # compute metrics
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {
        "wer": wer,
        "cer": cer,
    }

#### Sample run

In [ ]:
import torch

device = torch.device("cuda:0")

# move model
model.to(device)
samples = list(dataset["train"].take(2))

collated = data_collator(samples)
# move batch to GPU
batch = {k: v.to(device) for k, v in collated.items()}

with torch.no_grad():
    outputs = model(
        input_values=batch["input_values"],
        attention_mask=batch.get("attention_mask", None),
        labels=batch["labels"],  # optional, but keeps consistency
    )

# get logits
logits = outputs.logits.detach().cpu().numpy()
labels = batch["labels"].detach().cpu().numpy()

# wrap into object expected by compute_metrics
class Pred:
    def __init__(self, predictions, label_ids):
        self.predictions = predictions
        self.label_ids = label_ids

pred = Pred(predictions=logits, label_ids=labels)

# compute metrics
metrics = compute_metrics(pred)

print(metrics)

#### Training

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    # batch / optimization
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,   # effective batch = 16
    learning_rate=1e-4,
    
    # schedule
    warmup_ratio=0.01,
    lr_scheduler_type="cosine",
    max_steps=600,   # REQUIRED for streaming
    
    # precision
    fp16=True,
    
    # logging / saving
    save_steps=100,
    save_total_limit=2,
    
    # evaluation (optional)
    eval_strategy="steps",
    eval_steps=100,
    eval_accumulation_steps = 32,
    
    # misc
    report_to="none",   # or "wandb"
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    remove_unused_columns=True,
)

In [ ]:
train_dataset = dataset["train"]
eval_dataset = dataset["validation"]

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    processing_class=processor,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [8]:
processor.save_pretrained(MODEL_DIR)
model.save_pretrained(MODEL_DIR)

In [7]:
import torch
MODEL_DIR = "models/w2v2-lv-60-espeak-ipa/"
for l in LANGS:
    config = Wav2Vec2Config.from_pretrained(MODEL_DIR)
    config.adapter_attn_dim = 16
    model = Wav2Vec2ForCTC.from_pretrained(MODEL_DIR, config=config, ignore_mismatched_sizes=True)
    model.freeze_feature_encoder()

    adapter_state = {
        k: v.cpu()
        for k, v in model.state_dict().items()
        if ("adapter" in k) or ("lm_head" in k)
    }
    
    torch.save(
        adapter_state,
        f"{MODEL_DIR}/adapter.{l}.bin"
    )

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at models/w2v2-lv-60-espeak-ipa/ and are newly initialized: ['wav2vec2.encoder.layers.0.adapter_layer.linear_1.bias', 'wav2vec2.encoder.layers.0.adapter_layer.linear_1.weight', 'wav2vec2.encoder.layers.0.adapter_layer.linear_2.bias', 'wav2vec2.encoder.layers.0.adapter_layer.linear_2.weight', 'wav2vec2.encoder.layers.0.adapter_layer.norm.bias', 'wav2vec2.encoder.layers.0.adapter_layer.norm.weight', 'wav2vec2.encoder.layers.1.adapter_layer.linear_1.bias', 'wav2vec2.encoder.layers.1.adapter_layer.linear_1.weight', 'wav2vec2.encoder.layers.1.adapter_layer.linear_2.bias', 'wav2vec2.encoder.layers.1.adapter_layer.linear_2.weight', 'wav2vec2.encoder.layers.1.adapter_layer.norm.bias', 'wav2vec2.encoder.layers.1.adapter_layer.norm.weight', 'wav2vec2.encoder.layers.10.adapter_layer.linear_1.bias', 'wav2vec2.encoder.layers.10.adapter_layer.linear_1.weight', 'wav2vec2.encoder.layers.10.adapter_layer.linear_2.bias', 'wav2

In [5]:
import os
LANGS = sorted([d for d in os.listdir("fleurs_ipa_asr/") if '.' not in d])

In [2]:
!torchrun --nprocs 2 train.py --model-dir models/w2v2-lv-60-espeak-ipa --target-lang <lang> --learning-rate 1e-4 --max-steps 500 --warmup-ratio 0.01

usage: train.py [-h] --model-dir MODEL_DIR [--target-lang TARGET_LANG]
                [--batch-size BATCH_SIZE] [--learning-rate LEARNING_RATE]
                [--max-steps MAX_STEPS] [--warmup-ratio WARMUP_RATIO]

Train a Wav2Vec2 model for ASR on a given dataset.

options:
  -h, --help            show this help message and exit
  --model-dir MODEL_DIR
                        Pretrained model path from local directory. Processor
                        should also be present in the same directory.
                        Checkpoints will be saved in this directory.
  --target-lang TARGET_LANG
                        Valid individual languages are ['af_za', 'am_et',
                        'ar_eg', 'ast_es', 'az_az', 'be_by', 'bg_bg', 'bn_in',
                        'ca_es', 'ceb_ph', 'ckb_iq', 'cmn_hans_cn', 'cs_cz',
                        'cy_gb', 'da_dk', 'de_de', 'el_gr', 'en_us', 'es_419',
                        'et_ee', 'fa_ir', 'ff_sn', 'fi_fi', 'fr_fr', 'ga_ie',
           